# ABSA 추가 평가 — 같은 평가셋 재평가 · 도메인/출처별 분석 · 시연 데모

학습은 다시 하지 않고, **이미 저장된 모델 3개로 예측만** 합니다.

| 셀 | 내용 |
|---|---|
| 1 | 설정 (모델 폴더 경로 확인) |
| 2 | 데이터 파싱 — `absa_klue_roberta_v3`와 **완전히 같은 방식** (같은 검증셋을 재현하기 위해) |
| 3 | train / valid 분리 — v3와 같은 방식, 검증셋 전체 사용 |
| 4 | 예측 함수 |
| 5 | **A·B·C를 같은 검증셋 전체로 예측** (결과는 파일로 캐시) |
| 6 | 전체 성능표 + 원래 수치 재현 확인 |
| 7 | **도메인별 · 출처별(쇼핑몰/SNS) 성능** |
| 8 | **시연 데모**: 예시 리뷰 → 속성별 감정 + 확신도, 발표자료에 자동 삽입 |

실행 순서: 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8

In [ ]:
# 1. 설정
import os, glob, json, gc, re
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import numpy as np
import pandas as pd
import torch

DATA_DIR = "/home/jovyan/hallym_project2/Sample/02.라벨링데이터"   # v3 노트북과 같은 경로

# ▼ 저장된 모델 폴더 (아래 출력으로 실제 이름을 확인하고 맞추세요)
MODELS = {
    "A_baseline": "./absa_model",        # 베이스라인 (10만, Weighted CE)
    "B_focal":    "./absa_model_focal",  # 실험 1 (10만, Focal Loss)
    "C_full":     "./absa_model_full",   # 실험 2 = V3 (전체, Weighted CE)
}
MAX_LEN    = 512
BATCH_SIZE = 64      # 예측만 하므로 학습 때보다 크게. 메모리 부족하면 32
SEED       = 42
SUBSET_N   = 20000   # A·B를 원래 평가했던 검증셋 부분집합 크기 (MAX_SAMPLES 100000 // 5)
OUT_DIR    = "./eval_results"
os.makedirs(OUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

print("\n[현재 폴더의 모델 후보]")
for d in sorted(glob.glob("./absa_model*")):
    if os.path.isdir(d): print(" ", d)
missing = [k for k, p in MODELS.items() if not os.path.isfile(os.path.join(p, "config.json"))]
print("\n⚠ 경로를 찾지 못한 모델:", missing if missing else "없음")

json_files = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.json"), recursive=True))
print("JSON 파일 수:", len(json_files))

In [ ]:
# 2. 데이터 파싱 — v3 노트북 3번 셀과 같은 로직 (+ 도메인/출처 정보는 별도 표로 수집)
TEXT_KEY, ASPECTS_KEY, ASPECT_KEY, POLARITY_KEY = "RawText", "Aspects", "Aspect", "SentimentPolarity"
POL_MAP = {-1: "부정", 0: "중립", 1: "긍정"}

def norm_polarity(v):
    try:
        return POL_MAP.get(int(float(v)), str(v))
    except (TypeError, ValueError):
        return str(v).strip()

def iter_records(obj):
    if isinstance(obj, list):
        for x in obj: yield from iter_records(x)
    elif isinstance(obj, dict):
        if TEXT_KEY in obj:
            yield obj
        else:
            for v in obj.values():
                if isinstance(v, (list, dict)): yield from iter_records(v)

def split_of(path):
    p = path.lower()
    if "validation" in p or "valid" in p or "검증" in p: return "valid"
    return "train"

rows, meta, bad = [], {}, 0
for path in json_files:
    try:
        with open(path, encoding="utf-8-sig") as f:
            obj = json.load(f)
    except Exception:
        bad += 1; continue
    for rec in iter_records(obj):
        text = str(rec.get(TEXT_KEY, "")).strip()
        if text and text not in meta:   # 문장별 도메인·출처 (처음 나온 값)
            meta[text] = {"domain": rec.get("Domain"), "category": rec.get("MainCategory"), "source": rec.get("Source")}
        for a in rec.get(ASPECTS_KEY) or []:
            aspect = str(a.get(ASPECT_KEY, "")).strip()
            if text and aspect and a.get(POLARITY_KEY) is not None:
                rows.append({"text": text, "aspect": aspect,
                             "label": norm_polarity(a[POLARITY_KEY]), "split": split_of(path)})

df = pd.DataFrame(rows)
print(f"읽기 실패 파일: {bad} | 전체 (문장,속성) 쌍: {len(df):,}")

keys = ["text", "aspect", "split"]
cnt = df.groupby(keys + ["label"]).size().reset_index(name="n")
cnt = cnt.sort_values(keys + ["n"], ascending=[True, True, True, False])
cnt["rank"] = cnt.groupby(keys).cumcount()
first  = cnt[cnt["rank"] == 0]
second = cnt[cnt["rank"] == 1][keys + ["n"]].rename(columns={"n": "n2"})
m = first.merge(second, on=keys, how="left")
df = m[m["n2"].isna() | (m["n"] > m["n2"])][keys + ["label"]].reset_index(drop=True)
print(f"중복·충돌 정리 후: {len(df):,}")
del rows, cnt, first, second, m; gc.collect()

meta_df = pd.DataFrame.from_dict(meta, orient="index").rename_axis("text").reset_index()
del meta; gc.collect()
print(meta_df[["domain", "source"]].value_counts().head(12))

In [ ]:
# 3. train / valid 분리 — v3 노트북 4번 셀과 같은 방식 (MAX_SAMPLES 없이 검증셋 전체)
if (df["split"] == "valid").any() and (df["split"] == "train").any():
    train_df = df[df["split"] == "train"].reset_index(drop=True)
    valid_df = df[df["split"] == "valid"].reset_index(drop=True)
    print("분할 방식: AI Hub Training / Validation 폴더")
else:
    texts = np.array(df["text"].unique(), dtype=object)
    rng = np.random.default_rng(SEED)
    rng.shuffle(texts)
    valid_texts = set(texts[: int(len(texts) * 0.1)])
    is_valid = df["text"].isin(valid_texts)
    train_df, valid_df = df[~is_valid].reset_index(drop=True), df[is_valid].reset_index(drop=True)
    print("분할 방식: 문장 단위 9:1")

N_TRAIN_FULL = len(train_df)
print(f"train(전체) {N_TRAIN_FULL:,} | valid(전체) {len(valid_df):,}   ← 보고서의 '전체 학습 데이터 수'로 쓰세요")
print("기대값: valid 65,499 (다르면 DATA_DIR이나 데이터 버전이 학습 때와 다른 것)")

# A·B를 원래 평가했던 2만 개 부분집합 표시 (v3 노트북의 valid_df.sample과 같은 방식)
subset_idx = valid_df.sample(min(SUBSET_N, len(valid_df)), random_state=SEED).index
valid_df["in_subset"] = valid_df.index.isin(subset_idx)
valid_df = valid_df.merge(meta_df, on="text", how="left")
del train_df, df; gc.collect()
print(valid_df["label"].value_counts())

In [ ]:
# 4. 예측 함수 (모델별 id2label을 그대로 사용 → 라벨 순서가 달라도 안전)
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from tqdm.auto import tqdm

class PairDS(Dataset):
    def __init__(self, aspects, texts, tok):
        self.a, self.t, self.tok = aspects, texts, tok
    def __len__(self): return len(self.a)
    def __getitem__(self, i):
        return dict(self.tok(self.a[i], self.t[i], truncation="only_second",
                             max_length=MAX_LEN, return_token_type_ids=False))

@torch.no_grad()
def predict_pairs(model_dir, aspects, texts, desc=""):
    tok = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device).eval()
    id2label = {int(k): v for k, v in model.config.id2label.items()}
    dl = DataLoader(PairDS(aspects, texts, tok), batch_size=BATCH_SIZE, shuffle=False,
                    collate_fn=DataCollatorWithPadding(tok), num_workers=0)
    use_amp = device.type == "cuda"
    amp_dtype = torch.bfloat16 if use_amp and torch.cuda.is_bf16_supported() else torch.float16
    probs = []
    for batch in tqdm(dl, desc=desc, leave=False):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
            logits = model(**batch).logits
        probs.append(torch.softmax(logits.float(), -1).cpu().numpy())
    probs = np.concatenate(probs)
    preds = [id2label[i] for i in probs.argmax(-1)]
    labels_in_order = [id2label[i] for i in range(len(id2label))]
    del model; gc.collect(); torch.cuda.empty_cache()
    return preds, probs, labels_in_order

In [ ]:
# 5. A·B·C를 같은 검증셋 전체로 예측 (이미 예측한 모델은 파일에서 불러와 건너뜀)
pred_path = os.path.join(OUT_DIR, "valid_predictions.csv")
if os.path.exists(pred_path):
    cached = pd.read_csv(pred_path)
    if len(cached) == len(valid_df):
        for c in cached.columns:
            if c.startswith("pred_"): valid_df[c] = cached[c].values
        print("캐시 불러옴:", [c for c in valid_df.columns if c.startswith("pred_")])

for name, path in MODELS.items():
    col = f"pred_{name}"
    if col in valid_df.columns:
        continue
    if not os.path.isfile(os.path.join(path, "config.json")):
        print(f"⚠ {name}: {path} 없음 → 건너뜀"); continue
    preds, _, _ = predict_pairs(path, valid_df["aspect"].tolist(), valid_df["text"].tolist(), desc=name)
    valid_df[col] = preds
    valid_df.to_csv(pred_path, index=False)      # 모델 하나 끝날 때마다 저장
    print(f"{name} 완료 → {pred_path}")

In [ ]:
# 6. 전체 성능표 (같은 검증셋) + 원래 2만 개 부분집합에서의 재현 확인
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix

L = ["긍정", "부정", "중립"]
def scores(frame, col):
    g, p = frame["label"], frame[col]
    P, R, F, _ = precision_recall_fscore_support(g, p, labels=L, zero_division=0)
    return {"n": len(frame), "accuracy": accuracy_score(g, p), "macro_f1": f1_score(g, p, labels=L, average="macro"),
            "f1_긍정": F[0], "f1_부정": F[1], "f1_중립": F[2], "P_중립": P[2], "R_중립": R[2]}

pred_cols = [c for c in valid_df.columns if c.startswith("pred_")]
full = pd.DataFrame({c[5:]: scores(valid_df, c) for c in pred_cols}).T
sub  = pd.DataFrame({c[5:]: scores(valid_df[valid_df["in_subset"]], c) for c in pred_cols}).T

print("■ 검증셋 전체 (모든 모델 같은 조건)")
display(full.round(4))
print("■ 원래 평가했던 2만 개 부분집합 (A 0.8346, B 0.8355와 같게 나오면 재현 성공)")
display(sub.round(4))

if "pred_A_baseline" in valid_df and "pred_C_full" in valid_df:
    d = full.loc["C_full"] - full.loc["A_baseline"]
    print(f"\nC − A (같은 검증셋): macro-F1 {d['macro_f1']*100:+.2f}%p | 중립 F1 {d['f1_중립']*100:+.2f}%p")

for c in pred_cols:
    cm = pd.DataFrame(confusion_matrix(valid_df["label"], valid_df[c], labels=L), index=L, columns=L)
    print(f"\n[{c[5:]}] 혼동행렬 (행=정답, 열=예측)"); print(cm)

full.round(4).to_csv(os.path.join(OUT_DIR, "scores_full_valid.csv"))
sub.round(4).to_csv(os.path.join(OUT_DIR, "scores_subset20k.csv"))

In [ ]:
# 7. 도메인별 · 출처별 성능
def group_table(by):
    out = []
    for key, g in valid_df.groupby(by):
        row = {by: key, "n": len(g), "중립 수": int((g["label"] == "중립").sum())}
        for c in pred_cols:
            s = scores(g, c)
            row[f"{c[5:]} macroF1"] = s["macro_f1"]
            row[f"{c[5:]} 중립F1"]  = s["f1_중립"]
        out.append(row)
    return pd.DataFrame(out).sort_values("n", ascending=False).reset_index(drop=True)

by_domain = group_table("domain")
by_source = group_table("source")
print("■ 도메인별"); display(by_domain.round(4))
print("■ 출처별 (쇼핑몰 / SNS)"); display(by_source.round(4))

# 해석에 쓸 한 줄 요약: C 기준 가장 약한 도메인
if "C_full macroF1" in by_domain:
    w = by_domain.sort_values("C_full macroF1").iloc[0]
    print(f"\nC 기준 가장 낮은 도메인: {w['domain']} (macro-F1 {w['C_full macroF1']:.4f}, n={w['n']:,})")

by_domain.round(4).to_csv(os.path.join(OUT_DIR, "scores_by_domain.csv"), index=False)
by_source.round(4).to_csv(os.path.join(OUT_DIR, "scores_by_source.csv"), index=False)

## 8. 시연 데모

예시 리뷰를 V3(C) 모델에 넣어 속성별 감정과 확신도를 구합니다.
- 결과는 `eval_results/demo_results.json`으로 저장되고, 아래에 JSON이 출력됩니다.
- `DECK_PATH`에 발표자료 html 경로를 적으면 **데모 슬라이드에 결과가 자동으로 들어갑니다.** (서버에 발표자료를 올려 두고 실행한 뒤, 다시 내려받으세요)
- 속성 이름은 학습 데이터에 있는 이름을 쓰세요 (v3 노트북 3번 셀의 '상위 속성' 출력 참고).

In [ ]:
# 8. 시연 데모
DEMO_MODEL = MODELS["C_full"]
DEMO = [
    ("배송은 엄청 빨랐는데 가격 대비 품질이 너무 별로예요. 디자인은 예쁘네요.", ["배송", "가격", "품질", "디자인"]),
    ("소음은 그냥 보통이고 바람은 시원해요. 가격은 좀 있는 편이에요.",            ["소음", "기능", "가격"]),
    ("색감 무난무난~ 사이즈는 딱 맞음 근데 마감이 좀 아쉬움ㅠ",                    ["색상", "사이즈", "품질"]),
]

aspects = [a for _, asp in DEMO for a in asp]
texts   = [t for t, asp in DEMO for _ in asp]
preds, probs, order = predict_pairs(DEMO_MODEL, aspects, texts, desc="demo")

demo_out, k = [], 0
for text, asp in DEMO:
    res = []
    for a in asp:
        res.append({"aspect": a, "pred": preds[k],
                    "probs": {lab: round(float(probs[k][i]), 3) for i, lab in enumerate(order)}})
        k += 1
    demo_out.append({"review": text, "results": res})

for d in demo_out:
    print("\n" + d["review"])
    print(pd.DataFrame([{"속성": r["aspect"], "예측": r["pred"], **r["probs"]} for r in d["results"]]).to_string(index=False))

demo_json = json.dumps(demo_out, ensure_ascii=False)
with open(os.path.join(OUT_DIR, "demo_results.json"), "w", encoding="utf-8") as f:
    f.write(demo_json)
print("\n▼ 이 JSON을 그대로 복사해 두세요\n" + demo_json)

# 발표자료에 자동 삽입 (선택)
DECK_PATH = "./20236757_이시우_발표자료.html"
if os.path.exists(DECK_PATH):
    html = open(DECK_PATH, encoding="utf-8").read()
    new_html, n = re.subn(r'(<script type="application/json" id="demo-data">).*?(</script>)',
                          lambda m: m.group(1) + demo_json + m.group(2), html, flags=re.S)
    if n == 1:
        open(DECK_PATH, "w", encoding="utf-8").write(new_html)
        print(f"\n✅ {DECK_PATH} 데모 슬라이드에 결과를 넣었습니다.")
    else:
        print("\n⚠ 발표자료에서 demo-data 위치를 찾지 못했습니다.")
else:
    print(f"\n(발표자료 {DECK_PATH} 없음 → JSON만 저장)")